In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
import numpy as np
import cv2
import openslide
from glob import glob
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision
#from UNet.utilities.model import build_model
import matplotlib.pyplot as plt
#from UNet.utilities.metrics import dice_loss, dice_coef, iou
#from UNet.utilities.pre_processing import load_data, tf_dataset, read_image, read_mask
from tqdm import tqdm
import xml.etree.ElementTree as ET
from scipy.ndimage import gaussian_filter
from tensorflow.keras.models import load_model
from skimage.morphology import area_opening, area_closing
from shapely.geometry import Polygon
from skimage.measure import find_contours

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
from shapely.geometry import Polygon
import numpy as np
import cv2 as cv

def suppress_overlapping_regions(binary_mask, iou_threshold=0.5):
    """
    Suppress overlapping regions in a binary mask using IoU threshold.
    Keeps the larger region in case of overlap.
    """
    suppressed_mask = np.zeros_like(binary_mask, dtype=np.uint8)
    contours, _ = cv.findContours(binary_mask.astype(np.uint8), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    polygons = [(cv.contourArea(cnt), Polygon(cnt.squeeze())) for cnt in contours if len(cnt) >= 3]

    # Sort by area (large to small)
    polygons.sort(reverse=True, key=lambda x: x[0])
    retained_polys = []

    for area, poly in polygons:
        if not poly.is_valid:
            continue
        keep = True
        for _, existing_poly in retained_polys:
            if poly.intersects(existing_poly):
                iou = poly.intersection(existing_poly).area / poly.union(existing_poly).area
                if iou > iou_threshold:
                    keep = False
                    break
        if keep:
            retained_polys.append((area, poly))

    # Fill retained polygons in suppressed_mask
    for _, poly in retained_polys:
        if not poly.is_empty and poly.is_valid:
            int_coords = np.array(poly.exterior.coords, dtype=np.int32).reshape((-1, 1, 2))
            cv.drawContours(suppressed_mask, [int_coords], -1, 1, -1)

    return suppressed_mask

def remove_ptcs_overlapping_with_glom_artery(ptc_mask, pred_mask, overlap_threshold=0.1):
    """
    Remove PTC instances that overlap with glomerulus (1) or artery (2) based on mask overlaps.
    """
    filtered_mask = np.zeros_like(ptc_mask, dtype=np.uint8)

    # Get contours of glom and artery from pred_mask
    glom_mask = (pred_mask == 1).astype(np.uint8)
    artery_mask = (pred_mask == 2).astype(np.uint8)
    glom_contours, _ = cv.findContours(glom_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    artery_contours, _ = cv.findContours(artery_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    glom_polygons = [Polygon(cnt.squeeze()) for cnt in glom_contours if len(cnt) >= 3]
    artery_polygons = [Polygon(cnt.squeeze()) for cnt in artery_contours if len(cnt) >= 3]

    ptc_contours, _ = cv.findContours(ptc_mask.astype(np.uint8), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    for cnt in ptc_contours:
        if len(cnt) < 3:
            continue
        ptc_poly = Polygon(cnt.squeeze())
        if not ptc_poly.is_valid:
            continue
        keep = True
        for glom_poly in glom_polygons:
            if ptc_poly.intersects(glom_poly):
                if ptc_poly.intersection(glom_poly).area / ptc_poly.area > overlap_threshold:
                    keep = False
                    break
        for art_poly in artery_polygons:
            if ptc_poly.intersects(art_poly):
                if ptc_poly.intersection(art_poly).area / ptc_poly.area > overlap_threshold:
                    keep = False
                    break
        if keep:
            cv.drawContours(filtered_mask, [cnt], -1, 1, -1)

    return filtered_mask



In [ ]:
import openslide
import cv2 as cv
import numpy as np
import xml.etree.ElementTree as ET
from scipy.ndimage import gaussian_filter
from skimage.morphology import area_opening, area_closing
from skimage.measure import find_contours
from shapely.geometry import Polygon, Point
from tqdm import tqdm

def parse_roi_xml(xml_path, target_id="1"):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    polygons = []
    for annotation in root.findall(".//Annotation"):
        if str(annotation.attrib.get("Id")).strip() != str(target_id).strip():
            continue
        for region in annotation.find("Regions").findall("Region"):
            coords = [
                (float(v.attrib['X']), float(v.attrib['Y']))
                for v in region.find("Vertices").findall("Vertex")
            ]
            if len(coords) >= 3:
                polygons.append(Polygon(coords))
    return polygons

def detect_dab_regions_with_padding(tile_rgb, pad=32):
    padded_tile = cv.copyMakeBorder(tile_rgb, pad, pad, pad, pad, cv.BORDER_REFLECT_101)

    def convert_to_optical_densities(rgb):
        rgb = rgb.astype(float) + 1e-8
        OD = np.zeros_like(rgb, dtype=np.float32)
        OD[:, :, 0] = rgb[:, :, 0]
        OD[:, :, 1] = rgb[:, :, 1]
        OD[:, :, 2] = rgb[:, :, 2]
        return -np.log(OD)

    def color_deconvolution(rgb):
        stain_OD = np.array([[0.307, 0.537, 0.786], [0.157, 0.837, 0.525], [255, 255, 255]])
        normalized_OD = stain_OD / np.linalg.norm(stain_OD, axis=1, keepdims=True)
        D = np.linalg.inv(normalized_OD)
        OD = convert_to_optical_densities(rgb / 255.0)
        return np.reshape(np.dot(OD.reshape(-1, 3), D), OD.shape)

    C = color_deconvolution(padded_tile)
    brown_channel = C[:, :, 0] - C[:, :, 1]
    mask = cv.inRange(brown_channel, 0.6, 5)
    mask = gaussian_filter(mask, 1)
    _, thresh = cv.threshold(mask, 50, 255, cv.THRESH_BINARY)
    opened = area_opening(thresh, area_threshold=15, connectivity=1)
    closed = area_closing(opened, area_threshold=15, connectivity=1)
    dilated = cv.dilate(closed.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=1)
    filtered = area_opening(dilated, area_threshold=2, connectivity=1)

    h, w = filtered.shape
    return filtered[pad:h - pad, pad:w - pad]

def predict_and_save_xml(model_glom_artery, model_ptc, slide_path, xml_output_path,
                         level=2, scale_factor=None, xml_mask_path=None, overlap_fraction=0.5):
    if scale_factor is None:
        slide = openslide.OpenSlide(slide_path)
        scale_factor = slide.level_downsamples[level]

    slide = openslide.OpenSlide(slide_path)
    downsample = slide.level_downsamples[level]
    w, h = slide.level_dimensions[level]
    polygons = parse_roi_xml(xml_mask_path, target_id="1") if xml_mask_path else []

    tile_size = 224
    stride = int(tile_size * (1 - overlap_fraction))

    full_probs = np.zeros((h, w, 3), dtype=np.float32)
    full_ptc = np.zeros((h, w), dtype=np.uint8)
    full_dab_mask_sum = np.zeros((h, w), dtype=np.uint16)
    dab_count_map = np.zeros((h, w), dtype=np.float32)
    pred_count_map = np.zeros((h, w), dtype=np.float32)

    for y in tqdm(range(0, h - tile_size + 1, stride), desc="Scanning WSI"):
        for x in range(0, w - tile_size + 1, stride):
            center = Point((x + tile_size // 2) * downsample, (y + tile_size // 2) * downsample)
            if xml_mask_path and not any(poly.contains(center) for poly in polygons):
                continue

            region = slide.read_region((int(x * downsample), int(y * downsample)), level, (tile_size, tile_size)).convert("RGB")
            img = np.array(region)[:, :, :3]

            # Glom/Artery prediction
            img_glom = cv.resize(img, (512, 512)) / 255.0
            input_glom = np.expand_dims(img_glom.astype(np.float32), axis=0)
            probs = model_glom_artery.predict(input_glom, verbose=0)[0]
            probs = np.stack([gaussian_filter(probs[..., i], sigma=2.0) for i in range(3)], axis=-1)
            probs_resized = cv.resize(probs, (tile_size, tile_size), interpolation=cv.INTER_LINEAR)
            full_probs[y:y + tile_size, x:x + tile_size, :] += probs_resized
            pred_count_map[y:y + tile_size, x:x + tile_size] += 1

            # PTC prediction using 224x224 tiles (no resizing)
            img_ptc = img / 255.0
            input_ptc = np.expand_dims(img_ptc.astype(np.float32), axis=0)
            ptc_pred = model_ptc.predict(input_ptc, verbose=0)[0]
            ptc_mask = (ptc_pred.squeeze() > 0.5).astype(np.uint8)
            full_ptc[y:y + tile_size, x:x + tile_size] = np.maximum(full_ptc[y:y + tile_size, x:x + tile_size], ptc_mask)

            # DAB detection
            dab_mask = detect_dab_regions_with_padding(img, pad=32)
            full_dab_mask_sum[y:y + tile_size, x:x + tile_size] += dab_mask.astype(np.uint16)
            dab_count_map[y:y + tile_size, x:x + tile_size] += (dab_mask > 0).astype(np.float32)

    pred_count_map[pred_count_map == 0] = 1
    full_probs /= pred_count_map[..., np.newaxis]
    pred_mask = np.argmax(full_probs, axis=-1).astype(np.uint8)

    for class_id, min_area_thresh in [(1, 1500), (2, 500)]:
        class_mask = (pred_mask == class_id).astype(np.uint8)
        num_labels, labels, stats, _ = cv.connectedComponentsWithStats(class_mask, connectivity=8)
        for i in range(1, num_labels):
            if stats[i, cv.CC_STAT_AREA] < min_area_thresh:
                pred_mask[labels == i] = 0

    full_ptc[np.isin(pred_mask, [1, 2])] = 0

    dab_count_map[dab_count_map == 0] = 1
    normalized_dab = (full_dab_mask_sum / dab_count_map).astype(np.uint8)
    normalized_dab = (normalized_dab > 127).astype(np.uint8) * 255

    contours, _ = cv.findContours(normalized_dab, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    dab_layers = {4: [], 5: []}
    for contour in contours:
        area_px = cv.contourArea(contour)
        area_microns = area_px * (0.2281 ** 2)
        if area_microns > 1500:
            dab_layers[4].append(contour)
        else:
            dab_layers[5].append(contour)

    save_multiclass_mask_to_aperio_xml(pred_mask, full_ptc, dab_layers, xml_output_path, scale_factor)

    print("✅ Completed WSI scan using 224x224 overlapping tiles for both PTC and DAB.")


def save_multiclass_mask_to_aperio_xml(pred_mask, ptc_mask, dab_layers, output_xml_path,
                                       scale_factor=4.0, microns_per_pixel=0.2281):
    Annotations = ET.Element("Annotations", MicronsPerPixel=str(microns_per_pixel))
    color_map = {1: "65280", 2: "255", 3: "16711935", 4: "65535", 5: "16776960"}

    for class_id in [1, 2]:
        contours = find_contours((pred_mask == class_id).astype(np.uint8), level=0.5)
        annotation = ET.SubElement(Annotations, "Annotation", {
            "Id": str(class_id), "LineColor": color_map[class_id],
            "Type": "4", "Visible": "1"
        })
        regions = ET.SubElement(annotation, "Regions")
        for region_id, contour in enumerate(contours):
            scaled = np.array([[x * scale_factor, y * scale_factor] for y, x in contour])
            region = ET.SubElement(regions, "Region", {
                "Id": str(region_id), "Type": "0", "NegativeROA": "0", "DisplayId": str(region_id)
            })
            vertices = ET.SubElement(region, "Vertices")
            for x, y in scaled:
                ET.SubElement(vertices, "Vertex", {"X": str(x), "Y": str(y), "Z": "0"})

    contours = find_contours(ptc_mask.astype(np.uint8), level=0.5)
    annotation = ET.SubElement(Annotations, "Annotation", {
        "Id": "3", "LineColor": color_map[3], "Type": "4", "Visible": "1"
    })
    regions = ET.SubElement(annotation, "Regions")
    for region_id, contour in enumerate(contours):
        scaled = np.array([[x * scale_factor, y * scale_factor] for y, x in contour])
        region = ET.SubElement(regions, "Region", {
            "Id": str(region_id), "Type": "0", "NegativeROA": "0", "DisplayId": str(region_id)
        })
        vertices = ET.SubElement(region, "Vertices")
        for x, y in scaled:
            ET.SubElement(vertices, "Vertex", {"X": str(x), "Y": str(y), "Z": "0"})

    for class_id in [4, 5]:
        annotation = ET.SubElement(Annotations, "Annotation", {
            "Id": str(class_id), "LineColor": color_map[class_id], "Type": "4", "Visible": "1"
        })
        regions = ET.SubElement(annotation, "Regions")
        for region_id, contour in enumerate(dab_layers[class_id]):
            region = ET.SubElement(regions, "Region", {
                "Id": str(region_id), "Type": "0", "NegativeROA": "0", "DisplayId": str(region_id)
            })
            vertices = ET.SubElement(region, "Vertices")
            for point in contour[:, 0, :]:
                x, y = point * scale_factor
                ET.SubElement(vertices, "Vertex", {"X": str(x), "Y": str(y), "Z": "0"})

    tree = ET.ElementTree(Annotations)
    tree.write(output_xml_path)
    print(f"✅ Saved Aperio XML to: {output_xml_path}")


In [ ]:
model = load_model("/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Model_Development/Glom_Model_UNet/transunet_multiclass_denseNet_dice_newdata.h5", compile=False)
model_ptc = load_model("/orange/pinaki.sarder/durgasrit.dongla/PTC_IPTCR/Model/best_model.h5", compile=False)

In [ ]:
predict_and_save_xml(
    model_glom_artery=model,
    model_ptc=model_ptc,
    slide_path="/orange/pinaki.sarder/Davy_Jones_Locker/Transplant_R01/MVI_Dual_Stained(Acute)/Final_Analysis/S16-7255.1.A.5 - 2016-11-13 20.46.44.ndpi",
    xml_output_path="/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/DAB_new/S16-7255.1.A.5 - 2016-11-13 20.46.44.xml",
    xml_mask_path = "/blue/pinaki.sarder/akshitagupta/MicrovascularInf/Durga/ROI_XML/S16-7255.1.A.5 - 2016-11-13 20.46.44.xml"
)